# Lab 1: The Guessing Machine 🎲

You're going to build a machine that **guesses the next word**, using nothing but counting.
GPT does the same job (guess the next piece of text). It's just much, much bigger.

**How to use this notebook:** click a gray code cell and press **Shift + Enter** to run it.
Run the cells in order, from top to bottom.

If the first code cell gives an error, make sure you opened this notebook from the
`lessons/01-the-guessing-machine/lab/` folder.

In [ ]:
import math
import random

from guessing_machine import load_story, split_into_words, count_pairs, next_word_chances, generate

print("Ready! 🚀")

## Step 1: Chop the story into words

Computers can't "read". First we turn the story into a list of words.
We make everything lowercase, and we count each period `.` as a word too, so the machine
can learn when sentences end.

In [ ]:
text = load_story()
print(text)

words = split_into_words(text)
print(words[:12])
print("Number of words:", len(words))
print("Number of DIFFERENT words:", len(set(words)))

## Step 2: Predict, THEN count

✋ **Before you run the next cell:** look at the story above.
After the word **the**, which word do you think comes most often? How many times?

Type your guess between the quotes, then run the cell.

In [ ]:
my_guess = "???"   # for example: "dog 4"
print("My guess:", my_guess)

In [ ]:
counts = count_pairs(words)

print("After 'the' come:", counts["the"])
print("After 'cat' come:", counts["cat"])
print("After 'sat' come:", counts["sat"])

Were you right? 🤔 This is the big idea from Andrey Markov in 1913:
**don't argue about it, count it.**

Try changing `"cat"` above to another word from the story and run the cell again.

## Step 3: Turn counts into chances

If `the` was followed by 19 words, and 5 of them were `cat`, then the chance of `cat`
is 5 out of 19, or `5/19`.

Something **has** to come next, so all the chances should add up to exactly 1.

In [ ]:
chances = next_word_chances(counts, "the")
for word, chance in sorted(chances.items(), key=lambda pair: -pair[1]):
    print(f"{word:>5}: {chance:.3f}")

total = sum(chances.values())
print("Total:", total)
print("Is it (super close to) 1?", math.isclose(total, 1))

Why `math.isclose` and not `==`? Computers store decimals with tiny rounding crumbs.
Run this and see:

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(math.isclose(0.1 + 0.2, 0.3))

## Step 4: Let the machine talk

Now the machine picks each next word by rolling **weighted dice**: `cat` gets 5 "sides",
`log` gets only 1. Run this cell a few times.

In [ ]:
for _ in range(5):
    print(" ".join(generate(counts, "the", 12)))

Different every time! Is that a bug? 🐛 **No.** Rolling dice is the whole point.
But random doesn't mean "anything goes": every pair of neighbor words came from the story.

Want the *same* "random" sentence every time? Give it a **seed**:

In [ ]:
print(" ".join(generate(counts, "the", 12, rng=random.Random(42))))
print(" ".join(generate(counts, "the", 12, rng=random.Random(42))))

## Step 5: Teach it your own sentences

Add sentences of your own and see how the machine changes.
Keep them short, and end each one with a period.

In [ ]:
my_sentences = """
The cat ate the cake.
The dog chased the bird.
"""

bigger_story = text + "\n" + my_sentences
my_counts = count_pairs(split_into_words(bigger_story))

print("Now after 'the' come:", my_counts["the"])
for _ in range(5):
    print(" ".join(generate(my_counts, "the", 12)))

💬 **Talk about it:** Does the machine *know* anything about cats? Or is it just
repeating patterns it counted? What happens if you add a sentence that isn't true?

---

## 🧑‍🔬 Grown-up corner: surprise, bits, and GPT's loss

*(Optional. For older students and adults.)*

How likely is the whole sentence "the cat sat on the mat ." if we start from "the"?
Multiply the chances of each step (the **chain rule** with a one-word memory):

$$P = P(\text{cat}\mid\text{the})\,P(\text{sat}\mid\text{cat})\,P(\text{on}\mid\text{sat})\,P(\text{the}\mid\text{on})\,P(\text{mat}\mid\text{the})\,P(\text{.}\mid\text{mat})$$

**Surprise** of an event with chance $p$ is $-\log_2 p$ bits. Rare things are surprising.

In [ ]:
def sentence_chance(counts, sentence):
    ws = sentence.split()
    p = 1.0
    for a, b in zip(ws, ws[1:]):
        p *= next_word_chances(counts, a).get(b, 0.0)
    return p

for s in ["the cat sat on the mat .", "the dog sat on the log ."]:
    p = sentence_chance(counts, s)
    steps = len(s.split()) - 1
    bits = -math.log2(p)
    print(f"{s!r}: P = {p:.5f}, surprise = {bits:.3f} bits, "
          f"{bits / steps:.3f} bits/word, {-math.log(p) / steps:.3f} nats/word")

The cat sentence comes out about 4× more likely than the dog-and-log one. Which TWO steps make the difference?

**The link to GPT:** when nanoGPT trains, the "loss" number it prints is the **average surprise
per token, measured in nats** (natural log instead of log base 2). Training means nudging
the model so the real next token is less surprising. Our counting machine reaches about 0.80 nats/word
on that one sentence. GPT does the same kind of scoring on billions of tokens.

**Why logs? Underflow.** Multiply lots of small chances and the computer runs out of decimal places:

In [ ]:
p = 0.01
product = 1.0
for _ in range(400):
    product *= p
print("Multiplying 400 chances of 0.01:", product)          # rounds down to 0.0!
print("Adding 400 surprises instead:", 400 * -math.log2(p), "bits")  # no problem

---

## 🐞 Next: Bug Hunt!

Buddy, our AI helper, wrote its own versions of these functions, and **some of them have
mistakes planted on purpose**. Your job is to write tests that catch them.

Open a terminal in the `zero-to-gpt` folder and run:

```
python3 -m bughunt list
```